In [27]:
import pandas as pd

In [28]:
d=pd.read_csv('/content/Reviews.csv', on_bad_lines='skip')
df=pd.DataFrame(d)
df.head()

/tmp/ipykernel_1362/2643740501.py:1: DtypeWarning: Columns (4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  d=pd.read_csv('/content/Reviews.csv', on_bad_lines='skip')


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [29]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [30]:
df = df.drop(columns=['Id','ProfileName', 'HelpfulnessNumerator',
                     'HelpfulnessDenominator', 'Time', 'Summary', 'Text'], axis=1, errors='ignore')

In [31]:
df.head()

,ProductId,UserId,Score
0,B001E4KFG0,A3SGXH7AUHU8GW,5
1,B00813GRG4,A1D87F6ZCVE5NK,1
2,B000LQOCH0,ABXLMWJIXXAIN,4
3,B000UA0QIQ,A395BORC6FGVXV,2
4,B006K2ZZ7K,A1UQRSCLF8GW1T,5


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570484 entries, 0 to 570483
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   ProductId  570484 non-null  object
 1   UserId     570484 non-null  object
 2   Score      570483 non-null  object
dtypes: object(3)
memory usage: 13.1+ MB


In [34]:
df.dropna(inplace=True)

In [37]:
df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
df.dropna(subset=['Score'], inplace=True)
df['Score'] = df['Score'].astype(int)

In [41]:
import plotly.express as px
fig = px.histogram(df, x='Score', title='Distribution of Scores', color_discrete_sequence=['blue'])
fig.show()

In [50]:
top_items = df['ProductId'].value_counts().head(50).index
filtered_df = df[df['ProductId'].isin(top_items)]

fig = px.density_heatmap(filtered_df, x='ProductId', y='Score')
fig.show()

In [43]:
df.describe()

,Score
count,570482.000000
mean,4.183518
std,1.310170
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [44]:
num_users = df['UserId'].nunique()
num_items = df['ProductId'].nunique()
num_ratings = len(df)

max_possible_ratings = num_users * num_items
sparsity = (1 - (num_ratings / max_possible_ratings)) * 100

print(f"Number of Unique Users: {num_users:,}")
print(f"Number of Unique Items: {num_items:,}")
print(f"Number of Ratings: {num_ratings:,}")
print(f"Maximum Possible Connections: {max_possible_ratings:,}")
print(f"Sparsity: {sparsity:.6f}%")

Number of Unique Users: 256,058
Number of Unique Items: 74,257
Number of Ratings: 570,482
Maximum Possible Connections: 19,014,098,906
Dataset Sparsity: 99.997000%


In [45]:
import scipy.stats as stats


top_users = df['UserId'].value_counts().head(10).index
contingency_table = pd.crosstab(df[df['UserId'].isin(top_users)]['UserId'], df['Score'])

chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

print("=== Chi-Square Test of Independence (Top 10 Users vs. Scores) ===")
print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"p-value: {p:.4e}")
print(f"Degrees of Freedom: {dof}")
if p < 0.05:
    print("Result: Statistically significant. The rating distribution depends on the user (reject H0).")
else:
    print("Result: Not statistically significant. No evidence of dependence (fail to reject H0).")

=== Chi-Square Test of Independence (Top 10 Users vs. Scores) ===
Chi-Square Statistic: 3282.7347
p-value: 0.0000e+00
Degrees of Freedom: 36
Result: Statistically significant. The rating distribution depends on the user (reject H0).


In [46]:
user_counts = df['UserId'].value_counts()
item_counts = df['ProductId'].value_counts()


threshold = 5

cold_users = user_counts[user_counts < threshold].count()
cold_items = item_counts[item_counts < threshold].count()
print(f"Users with fewer than {threshold} reviews: {cold_users} ({cold_users / num_users * 100:.2f}% of total users)")
print(f"Items with fewer than {threshold} reviews: {cold_items} ({cold_items / num_items * 100:.2f}% of total items)")

print("\nUser Interaction Quantiles:")
print(user_counts.quantile([0.25, 0.5, 0.75, 0.9, 0.95]))
print("\nItem Interaction Quantiles:")
print(item_counts.quantile([0.25, 0.5, 0.75, 0.9, 0.95]))

=== Cold Start & Interaction Analysis ===
Users with fewer than 5 reviews: 232299 (90.72% of total users)
Items with fewer than 5 reviews: 53800 (72.45% of total items)

User Interaction Quantiles:
0.25    1.0
0.50    1.0
0.75    2.0
0.90    4.0
0.95    7.0
Name: count, dtype: float64

Item Interaction Quantiles:
0.25     1.0
0.50     2.0
0.75     5.0
0.90    14.0
0.95    26.0
Name: count, dtype: float64


In [51]:
dfsub=df[:5000]

In [57]:
px.histogram(dfsub, x='Score', title='Distribution of Scores', color_discrete_sequence=['blue'])

In [58]:
px.density_heatmap(dfsub, x='ProductId', y='Score')

In [59]:
dfsub.describe()

,Score
count,5000.000000
mean,4.150400
std,1.326549
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


Q1-the j shape is the closest to what i got from this amazon dataset

Q2-poisson

Q3-no rating entropy measures how evenly ratings are distributed

Q4-it made the item popularity more un equal so it doesnt represent the whole dataset

Q5-that user will have zero ratings in the subset depite having 20 or more

Q6-realistic user activity:use skewed dists so that some users rate many movies
realistic rating behaviour:generate ratings using user preferances